# Constructing party to lobby organization dataset

In [5]:
import pandas as pd
import numpy as np

In [ ]:
# 2019 - 2024
integrity_df_1 = pd.read_csv('../datasets/party/mepmeetings.csv')
integrity_df_1 = integrity_df_1[["country", "group", "lobbyists", "mep"]]
integrity_df_1.head()

,country,group,lobbyists,mep
0,Malta,S&D,Terumo Europe,Alex AGIUS SALIBA
1,Malta,S&D,"Tilray, Frank Farnel",Alex AGIUS SALIBA
2,Malta,S&D,Cannabis for Cancer Declaration,Alex AGIUS SALIBA
3,Malta,S&D,Eurocotrol,Alex AGIUS SALIBA
4,Malta,S&D,"Facebook, Phillip Malloch, Facebook Director f...",Alex AGIUS SALIBA


In [ ]:
# 2024 - 2029
integrity_df_2 = pd.read_json('../datasets/party/mepmeetings.json')
integrity_df_2 = integrity_df_2[["country", "group", "lobbyists", "mep"]]
integrity_df_2.head()

,country,group,lobbyists,mep
0,Finland,EPP,Konrad-Adenauer-Stiftung,Mika AALTOLA
1,Finland,EPP,International Institute for Democracy and Elec...,Mika AALTOLA
2,Finland,EPP,Elinkeinoelämän Keskusliitto,Mika AALTOLA
3,Finland,EPP,EPIS Thinktank,Mika AALTOLA
4,Finland,EPP,Konrad-Adenauer-Stiftung,Mika AALTOLA


In [8]:
integrity_df = integrity_df_1.merge(integrity_df_2)
integrity_df.head()

,country,group,lobbyists,mep
0,Malta,S&D,Terumo Europe,Alex AGIUS SALIBA
1,Malta,S&D,"Tilray, Frank Farnel",Alex AGIUS SALIBA
2,Malta,S&D,Cannabis for Cancer Declaration,Alex AGIUS SALIBA
3,Malta,S&D,Eurocotrol,Alex AGIUS SALIBA
4,Malta,S&D,"Facebook, Phillip Malloch, Facebook Director f...",Alex AGIUS SALIBA


In [9]:
# Make lobbyists an array
integrity_df['lobbyists'] = integrity_df['lobbyists'].apply(lambda x: np.array(list(map(str, x.split(',')))))
integrity_df.head()

,country,group,lobbyists,mep
0,Malta,S&D,[Terumo Europe],Alex AGIUS SALIBA
1,Malta,S&D,"[Tilray, Frank Farnel]",Alex AGIUS SALIBA
2,Malta,S&D,[Cannabis for Cancer Declaration],Alex AGIUS SALIBA
3,Malta,S&D,[Eurocotrol],Alex AGIUS SALIBA
4,Malta,S&D,"[Facebook, Phillip Malloch, Facebook Directo...",Alex AGIUS SALIBA


In [ ]:
# each lobbyist gets its own row
integrity_df_exploded = integrity_df.copy()
integrity_df_exploded['lobbyists'] = integrity_df_exploded['lobbyists'].apply(
    lambda x: [s.strip() for s in x] if isinstance(x, np.ndarray) else []
)
integrity_df_exploded = integrity_df_exploded.explode('lobbyists')

# Count meetings per lobbyist per EP group
lobby_group_counts = integrity_df_exploded.groupby(
    ['group', 'lobbyists']
).size().reset_index(name='meeting_count')

# print(lobby_group_counts.head())

# Compact set into only rows for each EP group with list of lobbyists and their meetings
lobby_group_summary = lobby_group_counts.groupby('group').agg({
    'lobbyists': lambda x: list(x),
    'meeting_count': lambda x: list(lobby_group_counts.loc[x.index, 'meeting_count'])
}).reset_index()

# Store total meetings per group as last column in lobby_group_summary
total_by_group = lobby_group_counts.groupby('group')['meeting_count'].sum().reset_index()
lobby_group_summary = lobby_group_summary.merge(total_by_group, on='group', how='left')

lobby_group_summary = lobby_group_summary.rename(columns={'meeting_count_y': 'total_meetings'})
lobby_group_summary = lobby_group_summary.rename(columns={'meeting_count_x': 'meetings_per_lobbyist'})


# Export to csv
lobby_group_summary.to_csv('../datasets/party/party_to_lobby_group.csv', index=False)

          group                                          lobbyists  \
0           ECR  [AIP - ASSOCIAZIONE ITALIANA PELLICCERIA, ASSO...   
1           EPP  [, "European Construction Industry Federation,...   
2       GUE/NGL  [ABP Coordinator, ACORN, ALTER-EU, ARD, ATTAC,...   
3  Greens / EFA  [, #FreeCourts Initiative, #SustainablePublicA...   
4            RE  [, (Deutscher Steuerberaterverband e.V.), 1&1,...   

                               meetings_per_lobbyist  total_meetings  
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...             197  
1  [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, ...            4284  
2  [1, 1, 1, 1, 5, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...             473  
3  [7, 5, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 3, 1, 2, ...            7580  
4  [1, 1, 1, 1, 1, 1, 23, 1, 3, 1, 3, 2, 1, 4, 1,...            4084  


In [13]:
lobby_group_summary.head()

,group,lobbyists,meetings_per_lobbyist,total_meetings
0,ECR,"[AIP - ASSOCIAZIONE ITALIANA PELLICCERIA, ASSO...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",197
1,EPP,"[, ""European Construction Industry Federation,...","[3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, ...",4284
2,GUE/NGL,"[ABP Coordinator, ACORN, ALTER-EU, ARD, ATTAC,...","[1, 1, 1, 1, 5, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",473
3,Greens / EFA,"[, #FreeCourts Initiative, #SustainablePublicA...","[7, 5, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 3, 1, 2, ...",7580
4,RE,"[, (Deutscher Steuerberaterverband e.V.), 1&1,...","[1, 1, 1, 1, 1, 1, 23, 1, 3, 1, 3, 2, 1, 4, 1,...",4084
